In [0]:
from pyspark.sql.types import StringType, StructField, StructType
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]
# Define schema using DDL string
schema_string = "id string, department_id string, name string, age string, gender string, salary string, hire_date string"

# Define schema using StructType
schema_spark = StructType([
    StructField("id", StringType(), True),
    StructField("department_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("age", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", StringType(), True),
    StructField("hire_date", StringType(), True)
])

emp = spark.createDataFrame(data=emp_data, schema=schema_spark)
display(emp)

In [0]:
# selections columns and 

from pyspark.sql.functions import expr
selected_columns = emp.select("id", expr("name"),col("age") ,emp.salary )
# display(selected_columns)

emp_casted=selected_columns.select(expr("id as employee_id"),expr( "cast (age as int) as age"   ), emp.salary)
# emp_casted.show()

emp_casted_1=selected_columns.selectExpr("id as employee_id","name as name","cast (age as int) as age","salary", "'defulat' as new_column")
emp_casted_1.show()

from pyspark.sql.functions import col

emp_final = emp_casted_1.select("name").where(col("salary") > 50000)


In [0]:
# implicit calling dataframe
schema_str="name string ,age int"
from pyspark.sql.types import _parse_datatype_string
new_schema=_parse_datatype_string(schema_str)

In [0]:
# Points for future studying:
# 1. You can define schema using DDL string or StructType; both are valid for Spark DataFrame creation.
# 2. Use selectExpr for concise column selection and transformation in one step.
# 3. Use expr or col for flexible column expressions and casting.
# 4. display() is preferred in Databricks for DataFrame visualization.
# 5. Use _parse_datatype_string for implicit schema parsing from string, which is convenient for quick schema definitions.
# 6. Manual schema definition (StructType) is best for complex or explicit schema requirements.

# Best way to call datatype:
# - Implicit (using _parse_datatype_string or DDL string) is best for simple, quick schema definitions.
# - Manual (using StructType) is best for complex schemas, explicit control, or when you need to specify nullable fields.

# Example: Implicit schema parsing
schema_str = "name string, age int"
from pyspark.sql.types import _parse_datatype_string
new_schema = _parse_datatype_string(schema_str)

# Example: Manual schema definition
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
manual_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

In [0]:
#07 Basic Transformation.
from pyspark.sql.functions import col , cast ,lit
df=emp.select("id","name","age",col("salary").cast("double"))
from pyspark.sql.functions import when

# new_df = df.withColumn(
#     "tax",
#     when(col("age") > 30, col("salary") * 0.2).otherwise(col("salary") * 0.04)
# )


df_taxed = df.withColumns({
    "tax": col("salary") * 0.2,
    "new_col": lit(1)
})
# display(df_taxed)

df_new = df.withColumns({
    "tax": when(col("age") > 30, col("salary") * 0.2).otherwise(col("salary") * 0.04),
    "bonus": when(col("salary") > 60000, lit(5000)).otherwise(lit(1000)),
    "is_senior": when(col("age") >= 35, lit(True)).otherwise(lit(False))
})
# display(df_new)
df_taxed=df.withColumn("tax",col("salary")*.2).withColumn("new_col",lit(1))
display(df_taxed)
#rename columns




In [0]:
from pyspark.sql.functions import col , cast ,lit

#rename columns
# emp.show()

emp_renamed=emp.withColumnRenamed("id" ,"emp_id")
emp_renamed=emp_renamed.withColumn("delete colum" ,lit(1))
emp_renamed=emp_renamed.drop("delete colum")
emp_renamed=emp_renamed.where(col("salary")>50000)
emp_renamed=emp_renamed.limit(10)

emp_renamed.show()


In [0]:
#basic string functions

#select emp_id,name,age,salary,gender
# gender male=m female =f
from pyspark.sql.functions import when, col
gender_fix = emp_renamed.withColumn("new_gender", when(col("gender") == "Male", "M").when(col("gender") == "Female", "F").otherwise("NA"))
# gender_fix.show()

from pyspark.sql.functions import expr

gender_fix_expr = emp_renamed.withColumn(
    "new_gender",
    expr("case when gender = 'Male' then 'M' when gender = 'Female' then 'F' else null end")
)
gender_fix_expr.show()

#reglaur experssion
from pyspark.sql.functions import regexp_replace

# Example: Remove non-digit characters from salary
emp_regex = emp_renamed.withColumn("salary_digits", regexp_replace(col("salary"), "[^0-9]", ""))

#convert date to_date
from pyspark.sql.functions import to_date ,current date ,current timestam

# Example: Convert hire_date to date type
emp_date = emp_renamed.withColumn("hire_date_dt", to_date(col("hire_date"), "yyyy-MM-dd"))
emp_date.show()


#drop null na drop, fix null values coalesce or nvl

# Drop rows with any null values
emp_no_nulls = emp_renamed.dropna()

# Fill null values in 'salary' column with 0 using coalesce
from pyspark.sql.functions import coalesce, lit
emp_fixed_nulls = emp_renamed.withColumn("salary", coalesce(col("salary"), lit("0")))

# Fill null values in 'salary' column with 0 using nvl (expr)
from pyspark.sql.functions import expr
emp_fixed_nulls_expr = emp_renamed.withColumn("salary", expr("nvl(salary, '0')"))

In [0]:
# Convert date column to string and extract year, month, day information
from pyspark.sql.functions import col, year, month, dayofmonth, date_format

# Assume emp_date DataFrame has 'hire_date_dt' as date type
emp_date_info = emp_date.withColumns({
    "hire_date_str": date_format(col("hire_date_dt"), "yyyy-MM-dd"),
    "year": year(col("hire_date_dt")),
    "month": month(col("hire_date_dt")),
    "day": dayofmonth(col("hire_date_dt"))
})
display(emp_date_info)
